# 01 数据清洗

数据来源：国家统计局 主要城市年度数据

https://data.stats.gov.cn/easyquery.htm?cn=E0105

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

DATA_RAW = r"D:\Project\City_Budget_Analysis\data_raw"
DATA_CLEAN = r"D:\Project\City_Budget_Analysis\data_clean"


## 1.1 读取原始数据

In [ ]:
# 读取合并后的原始数据
df = pd.read_csv(os.path.join(DATA_RAW, "merged_raw.csv"))
print(f"数据维度: {df.shape}")
print(f"城市数量: {df['city'].nunique()}")
print(f"年份范围: {df['year'].min()} - {df['year'].max()}")
print(f"\n城市列表:")
print(df['city'].unique())
df.head(10)


In [ ]:
# 查看数据基本信息
df.info()


In [ ]:
# 检查缺失值
print("各列缺失值统计:")
print(df.isnull().sum())
print(f"\n缺失值比例:")
print((df.isnull().sum() / len(df) * 100).round(2))


## 1.2 数据清洗

In [ ]:
# 将year转为整数
df['year'] = df['year'].astype(int)

# 过滤掉2025年(可能无数据)和无效数据
df = df[df['year'] <= 2024].copy()

# 检查各年份数据完整性
print("各年份城市数量:")
print(df.groupby('year')['city'].count())


In [ ]:
# 查看各指标的缺失情况(按年份)
for col in ['income', 'expend', 'gdp', 'deposit']:
    missing = df[df[col].isnull()].groupby('year')['city'].count()
    if len(missing) > 0:
        print(f"\n{col} 缺失年份:")
        print(missing)
    else:
        print(f"{col}: 无缺失")


In [ ]:
# 删除income, expend, gdp全为空的行
df = df.dropna(subset=['income', 'expend', 'gdp'], how='all').copy()

# 计算预算缺口
df['gap'] = df['expend'] - df['income']
df['gap_to_gdp'] = df['gap'] / df['gdp']

print(f"清洗后数据维度: {df.shape}")
df.describe()


## 1.3 计算增长率

In [ ]:
# 按城市排序后计算年度增长率
df = df.sort_values(['city', 'year']).reset_index(drop=True)

for col in ['income', 'expend']:
    df[f'{col}_growth'] = df.groupby('city')[col].pct_change() * 100

print("增长率统计:")
print(df[['income_growth', 'expend_growth']].describe())


## 1.4 城市分组

In [ ]:
# 定义城市分组
tier1 = ['北京', '上海', '广州', '深圳']

# 珠三角城市(在36个主要城市中)
pearl_river = ['广州', '深圳', '珠海', '佛山', '东莞', '中山', '惠州']
# 检查哪些在数据中
pearl_river_available = [c for c in pearl_river if c in df['city'].unique()]
print(f"珠三角城市(数据中可用): {pearl_river_available}")

# 长三角城市(在36个主要城市中)
yangtze_river = ['上海', '南京', '杭州', '苏州', '无锡', '宁波', '合肥', '南通']
yangtze_river_available = [c for c in yangtze_river if c in df['city'].unique()]
print(f"长三角城市(数据中可用): {yangtze_river_available}")

# 添加分组标签
def assign_group(city):
    if city in pearl_river_available:
        return '珠三角'
    elif city in yangtze_river_available:
        return '长三角'
    else:
        return '其他'

df['region_group'] = df['city'].apply(assign_group)
df['is_tier1'] = df['city'].isin(tier1)

print(f"\n分组统计:")
print(df.groupby('region_group')['city'].nunique())


## 1.5 保存清洗后数据

In [ ]:
# 保存主数据
df.to_csv(os.path.join(DATA_CLEAN, "city_budget_clean.csv"), index=False, encoding="utf-8-sig")
print(f"主数据已保存: {df.shape}")

# 读取并保存房地产数据
df_re = pd.read_csv(os.path.join(DATA_RAW, "real_estate_raw.csv"))
df_re['year'] = df_re['year'].astype(int)
df_re = df_re[df_re['year'] <= 2024].copy()

# 重命名列
df_re.columns = ['city', 'year', 're_invest', 'sale_area', 're_avg_price', 
                  'res_sale_area', 'res_avg_price']
df_re.to_csv(os.path.join(DATA_CLEAN, "real_estate_clean.csv"), index=False, encoding="utf-8-sig")
print(f"房地产数据已保存: {df_re.shape}")

# 合并主数据和房地产数据
df_all = df.merge(df_re, on=['city', 'year'], how='left')
df_all.to_csv(os.path.join(DATA_CLEAN, "city_all_clean.csv"), index=False, encoding="utf-8-sig")
print(f"合并数据已保存: {df_all.shape}")
print(f"\n最终数据列: {df_all.columns.tolist()}")
df_all.head()
